In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Config ===
# Correlates per-object packaging metadata with measurement-point sampling clearance
# (Part A) and YOLO segmentation quality (Part B). BASE_DIR holds the input CSVs;
# result tables are written to RESULTS_DIR.
from pathlib import Path

BASE_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/06_Multimodal_Pipeline/20260625_Result_CSVs")  # directory containing the CSVs

META_CSV             = BASE_DIR / "20260617_Extended_Metadata.csv"
SAMPLING_RESULTS_CSV = BASE_DIR / "20260622_Measurement_Point_Sampling_sampling_results.csv"
GRID_POINTCOUNTS_CSV = BASE_DIR / "20260622_Measurement_Point_Sampling_grid_pointcounts_20260623_131617.csv"
SEG_PER_IMAGE_CSV    = BASE_DIR / "20260603_Yolo_Deployment_Training_per_image_region_metrics.csv"
SEG_PER_OBJECT_CSV   = BASE_DIR / "20260603_Yolo_Deployment_Training_per_object_region_metrics.csv"

META_SEP              = ";"                 # metadata file is semicolon-delimited
SENSOR_RADIUS_PX      = 44                  # sensor footprint radius; clearance >= this -> point fits
RANDOM_SEED           = 42
MIN_OBJECTS_PER_GROUP = 3                   # minimum objects for a group to enter an object-level test

PROJECT_NAME = "Metadata_Correlation"
RESULTS_DIR  = BASE_DIR / "results"

In [ ]:
# === Cell 3: Imports and Helpers ===
# Significance tests run at object level (one value per object).
import re
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
np.random.seed(RANDOM_SEED)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

OBJ_RE = re.compile(r"([A-Za-z]+_obj_\d+)")

def parse_object_name(stem):
    """Extract the canonical object name (e.g. 'HDPE_obj_01') from a file stem."""
    m = OBJ_RE.match(str(stem))
    return m.group(1) if m else None

def color_tokens(value):
    """Split a colour field into a set of lowercased tokens (comma/semicolon separated)."""
    if pd.isna(value):
        return set()
    return {t.strip().lower() for t in str(value).replace(";", ",").split(",") if t.strip()}

def share_token(a, b):
    """Colour match: 1 if the two token sets share >=1 colour, 0 if not, NaN if either empty."""
    if not a or not b:
        return np.nan
    return int(len(a & b) > 0)

def object_level_test(obj_df, group_col, value_col, min_objects=MIN_OBJECTS_PER_GROUP):
    """Kruskal-Wallis (>2 groups) or Mann-Whitney U (2 groups) on one value per object.

    Groups with fewer than min_objects rows are skipped.
    Returns (statistic, p_value, n_groups_used)."""
    d = obj_df.dropna(subset=[group_col, value_col])
    groups = [g[value_col].values for _, g in d.groupby(group_col) if len(g) >= min_objects]
    if len(groups) < 2:
        return np.nan, np.nan, len(groups)
    if len(groups) == 2:
        stat, p = stats.mannwhitneyu(*groups)
    else:
        stat, p = stats.kruskal(*groups)
    return float(stat), float(p), len(groups)

In [ ]:
# === Cell 4: Metadata Colour and Transparency Flags ===
meta = pd.read_csv(META_CSV, sep=META_SEP)

for part in ["body", "label", "cap"]:
    meta[f"{part}_tok"] = meta[f"{part}_color"].apply(color_tokens)

# Colour match = the two parts share at least one colour token
meta["label_body_color_match"] = meta.apply(lambda r: share_token(r["label_tok"], r["body_tok"]), axis=1)
meta["cap_body_color_match"]   = meta.apply(lambda r: share_token(r["cap_tok"],   r["body_tok"]), axis=1)
meta["body_transparent"]       = meta["body_color"].astype(str).str.contains("transparent", case=False, na=False).astype(int)

print(f"Metadata objects: {len(meta)}")
meta[["object_name", "body_class", "body_type", "body_transparent",
      "label_body_color_match", "cap_body_color_match"]].head()

In [ ]:
# === Cell 5: A.0 Load Sampling Results ===
# Part A (measurement-point sampling): analysis unit is primary, valid points
# (is_main & valid); clearance = score in px; a point fits if score >= SENSOR_RADIUS_PX.
samp = pd.read_csv(SAMPLING_RESULTS_CSV)
grid = pd.read_csv(GRID_POINTCOUNTS_CSV)
samp["object_name"] = samp["file"].apply(parse_object_name)
grid["object_name"] = grid["stem"].apply(parse_object_name)

samp = samp.merge(meta, on="object_name", how="left")
# Region-specific metadata (body_* for body rows, cap_* for caps, label_* for labels)
for field in ["class", "type", "color"]:
    samp[f"region_{field}"] = samp.apply(lambda r: r.get(f"{r['class_name']}_{field}", np.nan), axis=1)

M = samp[samp["is_main"] & samp["valid"]].copy()
M["fits"] = (M["score"] >= SENSOR_RADIUS_PX).astype(int)

print(f"Sampling rows: {len(samp)} | analysis points (is_main & valid): {len(M)}")
print(f"Objects: {M['object_name'].nunique()} | files: {M['file'].nunique()}")

In [ ]:
# === Cell 6: A.1 Sensor-Fit Rate by Region ===
# Fit rate and clearance per region x source x algorithm.
fit_tab = (M.groupby(["class_name", "source", "algorithm"])
             .agg(n=("fits", "size"),
                  fit_rate_pct=("fits", lambda x: round(100 * x.mean(), 1)),
                  clearance_median=("score", "median"),
                  clearance_mean=("score", "mean"))
             .round(1))
fit_tab.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_sampling_fit_by_region_algo.csv")
fit_tab

In [ ]:
# === Cell 7: A.2 Sampling Algorithm Effect ===
# EDT_PEAK vs CENTROID clearance per region: paired Wilcoxon test, points paired on
# (source, file, class_name, instance_idx).
key = ["source", "file", "class_name", "instance_idx"]
piv = (M.pivot_table(index=key, columns="algorithm", values="score")
         .dropna(subset=["CENTROID", "EDT_PEAK"]).reset_index())

algo_rows = []
for cls in ["body", "cap", "label"]:
    sub = piv[piv["class_name"] == cls]
    _, p = stats.wilcoxon(sub["EDT_PEAK"], sub["CENTROID"])
    algo_rows.append({"region": cls, "n_pairs": len(sub),
                      "median_CENTROID": round(sub["CENTROID"].median(), 1),
                      "median_EDT_PEAK": round(sub["EDT_PEAK"].median(), 1),
                      "median_delta": round((sub["EDT_PEAK"] - sub["CENTROID"]).median(), 1),
                      "mean_delta": round((sub["EDT_PEAK"] - sub["CENTROID"]).mean(), 1),
                      "wilcoxon_p": p})
algo_df = pd.DataFrame(algo_rows)
algo_df.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_sampling_algorithm_effect.csv", index=False)
algo_df

In [ ]:
# === Cell 8: A.3 Predicted vs GT Clearance ===
# Predicted vs ground-truth mask clearance per algorithm and region (Mann-Whitney U).
rob_rows = []
for algo in ["EDT_PEAK", "CENTROID"]:
    for cls in ["body", "cap", "label"]:
        g  = M[(M.class_name == cls) & (M.source == "gt")        & (M.algorithm == algo)]["score"]
        pr = M[(M.class_name == cls) & (M.source == "predicted") & (M.algorithm == algo)]["score"]
        _, pp = stats.mannwhitneyu(g, pr)
        rob_rows.append({"algorithm": algo, "region": cls,
                         "gt_median": round(g.median(), 1), "pred_median": round(pr.median(), 1),
                         "diff_pred_minus_gt": round(pr.median() - g.median(), 1),
                         "mwu_p": round(pp, 4)})
rob_df = pd.DataFrame(rob_rows)
rob_df.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_sampling_gt_vs_predicted.csv", index=False)
rob_df

In [ ]:
# === Cell 9: A.4 Region Area Effect ===
# (i) Instance-level Spearman of clearance vs region area; (ii) object-level metadata
# tests before and after regressing clearance on log(area). Uses EDT_PEAK / predicted.
P = M[(M.algorithm == "EDT_PEAK") & (M.source == "predicted")].copy()
P["log_area"] = np.log(P["valid_area_px2"].clip(lower=1))

print("(i) Spearman  clearance vs valid_area_px2:")
for cls in ["body", "cap", "label"]:
    s = P[P.class_name == cls]
    rho, p = stats.spearmanr(s["valid_area_px2"], s["score"])
    print(f"    {cls:6s} rho={rho:+.3f}  p={p:.2e}  n={len(s)}")

def metadata_clearance_tests(P, region):
    """Object-level raw vs area-residualised tests of metadata fields on clearance."""
    sub = P[P.class_name == region]
    obj = (sub.groupby("object_name")
              .agg(clearance=("score", "median"), log_area=("log_area", "median"))
              .reset_index()
              .merge(meta, on="object_name", how="left"))
    b, a = np.polyfit(obj["log_area"], obj["clearance"], 1)
    obj["clearance_resid"] = obj["clearance"] - (b * obj["log_area"] + a)
    fields = [f"{region}_type", f"{region}_class"]
    if region == "body":
        fields.append("body_transparent")
    rows = []
    for f in fields:
        _, p_raw, ng = object_level_test(obj, f, "clearance")
        _, p_res, _  = object_level_test(obj, f, "clearance_resid")
        rows.append({"region": region, "field": f, "n_groups": ng,
                     "p_raw": round(p_raw, 4) if p_raw == p_raw else np.nan,
                     "p_after_area": round(p_res, 4) if p_res == p_res else np.nan})
    return pd.DataFrame(rows)

partial_df = pd.concat([metadata_clearance_tests(P, r) for r in ["body", "cap", "label"]],
                       ignore_index=True)
partial_df.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_sampling_area_partial_tests.csv", index=False)
print("\n(ii) Object-level raw vs after-area p-values:")
partial_df

In [ ]:
# === Cell 10: A.5 Metadata Groups per Region ===
# Point-weighted summaries: body by material and type, label and cap by type.
def region_group_summary(P, region, group_col):
    """Per-group object/point counts, median clearance, fit rate and median area for one region."""
    sub = P[P.class_name == region]
    g = (sub.groupby(group_col)
            .agg(n_objects=("object_name", "nunique"),
                 n_points=("score", "size"),
                 clearance_median=("score", "median"),
                 fit_rate_pct=("fits", lambda x: round(100 * x.mean(), 1)),
                 area_median=("valid_area_px2", "median"))
            .round(1).sort_values("clearance_median"))
    return g

body_by_material = region_group_summary(P, "body", "body_class")
body_by_type     = region_group_summary(P, "body", "body_type")
label_by_type    = region_group_summary(P, "label", "label_type")
cap_by_type      = region_group_summary(P, "cap", "cap_type")

for name, tbl in [("sampling_body_by_material", body_by_material),
                  ("sampling_body_by_type", body_by_type),
                  ("sampling_label_by_type", label_by_type),
                  ("sampling_cap_by_type", cap_by_type)]:
    tbl.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_{name}.csv")

print("BODY by material:");  display(body_by_material)
print("BODY by type:");      display(body_by_type)
print("LABEL by type:");     display(label_by_type)
print("CAP by type:");       display(cap_by_type)

In [ ]:
# === Cell 11: A.6 Body Objects Ranked ===
# Body-region objects ranked by median clearance (EDT_PEAK / predicted).
body_obj = (P[P.class_name == "body"]
              .groupby("object_name")
              .agg(clearance_median=("score", "median"),
                   fit_rate_pct=("fits", lambda x: round(100 * x.mean())),
                   area_median=("valid_area_px2", "median"))
              .reset_index()
              .merge(meta[["object_name", "object_description", "body_class",
                           "body_type", "body_color"]], on="object_name")
              .sort_values("clearance_median"))
body_obj.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_sampling_objects_ranked.csv", index=False)

print("Weakest 8:"); display(body_obj.head(8))
print("Strongest 6:"); display(body_obj.tail(6))

In [ ]:
# === Cell 12: B.0 Load Segmentation Metrics ===
# Part B (YOLO segmentation): mask IoU over true-positive masks and TP/FP/FN outcomes.
# `material` is the filename prefix (HDPE/PET/PP/PS); other fields come from metadata.
seg_img = pd.read_csv(SEG_PER_IMAGE_CSV)
seg_obj = pd.read_csv(SEG_PER_OBJECT_CSV)
for df in (seg_img, seg_obj):
    df["object_name"] = df["material"] + "_obj_" + df["object_id"].astype(str).str.zfill(2)
seg_img = seg_img.merge(meta, on="object_name", how="left")
seg_obj = seg_obj.merge(meta, on="object_name", how="left")

print(f"per-image rows: {len(seg_img)} | per-object rows: {len(seg_obj)}")
print("match_status:", seg_img["match_status"].value_counts().to_dict())

In [ ]:
# === Cell 13: B Detection Summary Helpers ===
def detection_counts(df):
    """Return (TP, FP, FN) counts from match_status."""
    return (int((df["match_status"] == "TP").sum()),
            int((df["match_status"] == "FP").sum()),
            int((df["match_status"] == "FN").sum()))

def detection_summary(df, by):
    """TP/FP/FN, recall, precision and mean TP IoU per group of `by`."""
    by_cols = by if isinstance(by, list) else [by]
    rows = []
    for key, g in df.groupby(by):
        tp, fp, fn = detection_counts(g)
        recall    = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
        iou_tp    = g.loc[g["match_status"] == "TP", "iou"].mean()
        row = {"TP": tp, "FP": fp, "FN": fn,
               "recall_pct": round(100 * recall, 1) if recall == recall else np.nan,
               "precision_pct": round(100 * precision, 1) if precision == precision else np.nan,
               "IoU_TP": round(iou_tp, 3)}
        keyvals = key if isinstance(key, tuple) else (key,)
        for col, val in zip(by_cols, keyvals):
            row[col] = val
        rows.append(row)
    out = pd.DataFrame(rows)
    return out[by_cols + ["TP", "FP", "FN", "recall_pct", "precision_pct", "IoU_TP"]]

In [ ]:
# === Cell 14: B.1 Region Summary ===
# Detection and IoU summary per region.
region_summary = detection_summary(seg_img, "class_name").sort_values("IoU_TP", ascending=False)
region_summary.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_seg_region_summary.csv", index=False)
region_summary

In [ ]:
# === Cell 15: B.2 Region by Material ===
# Detection and IoU per region x material; the cap block lists only materials with
# cap ground truth.
rm = detection_summary(seg_img, ["class_name", "material"])
rm = rm[(rm["TP"] + rm["FN"]) > 0]   # region-material combos with ground truth only
order = {"body": 0, "cap": 1, "label": 2, "HDPE": 0, "PET": 1, "PP": 2, "PS": 3}
rm = rm.sort_values(by=["class_name", "material"], key=lambda s: s.map(order)).reset_index(drop=True)
rm.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_seg_region_material.csv", index=False)

# Object-level test of body metrics across materials
body_obj_seg = seg_obj[seg_obj.class_name == "body"]
_, p_iou, _ = object_level_test(body_obj_seg, "material", "iou_mean")
_, p_f1,  _ = object_level_test(body_obj_seg, "material", "f1")
print(f"Body IoU by material: object-level p={p_iou:.4f}")
print(f"Body F1  by material: object-level p={p_f1:.4f}")
rm

In [ ]:
# === Cell 16: B.3 Colour-Match Tests ===
# Object-level tests of label/cap segmentation metrics vs colour match with the body
# (label/cap and body colour fields share >= 1 token).
def colormatch_test(seg_obj, region, match_col):
    """Object-level tests of IoU, F1, recall and precision against a colour-match flag."""
    sub = seg_obj[seg_obj["class_name"] == region]
    rows = []
    for metric in ["iou_mean", "f1", "recall", "precision"]:
        _, p, ng = object_level_test(sub, match_col, metric)
        means = sub.groupby(match_col)[metric].mean().round(3).to_dict()
        rows.append({"region": region, "factor": match_col, "metric": metric,
                     "n_groups": ng,
                     "p_value": round(p, 4) if p == p else np.nan,
                     "group_means": means})
    return pd.DataFrame(rows)

colormatch_df = pd.concat([
    colormatch_test(seg_obj, "label", "label_body_color_match"),
    colormatch_test(seg_obj, "cap",   "cap_body_color_match"),
], ignore_index=True)
colormatch_df.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_seg_colormatch_tests.csv", index=False)
colormatch_df

In [ ]:
# === Cell 17: B.4 Label Errors by Transparency ===
# Label detection split by body transparency, object-level tests, and FP/FN counts
# on transparent bodies.
lab_by_transp = detection_summary(seg_img[seg_img.class_name == "label"], "body_transparent")
lab_by_transp.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_seg_label_by_transparency.csv", index=False)
print("Label detection by body transparency (0=opaque, 1=transparent):")
display(lab_by_transp)

lab_obj = seg_obj[seg_obj.class_name == "label"]
for metric in ["f1", "recall", "iou_mean"]:
    _, p, _ = object_level_test(lab_obj, "body_transparent", metric)
    print(f"  label {metric:9s} object-level p={p:.4f}")

lab_img = seg_img[seg_img.class_name == "label"]
fp_t  = int(lab_img[(lab_img.match_status == "FP")]["body_transparent"].sum())
fp_all = int((lab_img.match_status == "FP").sum())
fn_t  = int(lab_img[(lab_img.match_status == "FN")]["body_transparent"].sum())
fn_all = int((lab_img.match_status == "FN").sum())
print(f"\nLabel FP on transparent bodies: {fp_t}/{fp_all}")
print(f"Label FN on transparent bodies: {fn_t}/{fn_all}")

In [ ]:
# === Cell 18: List Saved Results ===
print("Saved result files:")
for f in sorted(RESULTS_DIR.glob(f"{PROJECT_NAME}_*.csv")):
    print("  ", f.name)